In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

print("DeepShield Research Environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device: CPU")

DeepShield Research Environment
Python: 3.14.10 (main, Nov  4 2026, 18:21:50) [GCC 16.2.1 20260810]
PyTorch: 2.15.0+cpu
Device: CPU


In [3]:
PROJECT_ROOT = Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project:", PROJECT_ROOT)
print("Raw data:", DATA_RAW)
print("Processed data:", DATA_PROCESSED)

Project: /home/darling/Projects/DeepShield
Raw data: /home/darling/Projects/DeepShield/data/raw
Processed data: /home/darling/Projects/DeepShield/data/processed


In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/home/darling/Projects/DeepShield/data/raw/CIC-IDS2017/"
    "MachineLearningCSV/MachineLearningCVE"
)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Total CSV files:", len(csv_files))

for file in csv_files:
    df = pd.read_csv(file, nrows=5)

    # Remove accidental whitespace around column names
    df.columns = df.columns.str.strip()

    print("\n" + "=" * 65)
    print("FILE:", file.name)
    print("NUMBER OF COLUMNS:", len(df.columns))
    print("LABEL COLUMN EXISTS:", "Label" in df.columns)
    print("FIRST 8 COLUMNS:", df.columns[:8].tolist())

    if "Label" in df.columns:
        print("SAMPLE LABELS:", df["Label"].unique())

Total CSV files: 8

FILE: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
NUMBER OF COLUMNS: 79
LABEL COLUMN EXISTS: True
FIRST 8 COLUMNS: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min']
SAMPLE LABELS: <StringArray>
['BENIGN']
Length: 1, dtype: str

FILE: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
NUMBER OF COLUMNS: 79
LABEL COLUMN EXISTS: True
FIRST 8 COLUMNS: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min']
SAMPLE LABELS: <StringArray>
['BENIGN']
Length: 1, dtype: str

FILE: Friday-WorkingHours-Morning.pcap_ISCX.csv
NUMBER OF COLUMNS: 79
LABEL COLUMN EXISTS: True
FIRST 8 COLUMNS: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of F

In [4]:
from collections import Counter

label_counts = Counter()

for file in csv_files:
    print("Scanning:", file.name)

    for chunk in pd.read_csv(
        file,
        usecols=lambda col: col.strip() == "Label",
        chunksize=100_000
    ):
        labels = chunk.iloc[:, 0].astype("string").str.strip()

        label_counts.update(
            labels.value_counts().to_dict()
        )

print("\n========== DATASET LABEL DISTRIBUTION ==========")

for label, count in label_counts.most_common():
    print(f"{label:<25} {count:>12,}")

print("\nTotal records:", sum(label_counts.values()))
print("Total unique labels:", len(label_counts))

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv

========== DATASET LABEL DISTRIBUTION ==========
BENIGN                       2,273,097
DoS Hulk                       231,073
PortScan                       158,930
DDoS                           128,027
DoS GoldenEye                   10,293
FTP-Patator                      7,938
SSH-Patator                      5,897
DoS slowloris                    5,796
DoS Slowhttptest                 5,499
Bot                              1,966
Web Attack � Brute Force         1,507
Web Attack � XSS                   652
Infiltration                        3

In [5]:
import pandas as pd

label_df = pd.DataFrame(
    label_counts.items(),
    columns=["Label", "Record_Count"]
)

label_df["Percentage"] = (
    label_df["Record_Count"] / label_df["Record_Count"].sum()
) * 100

label_df = label_df.sort_values(
    by="Record_Count",
    ascending=False
)

label_df.to_csv(
    "../reports/class_distribution.csv",
    index=False
)

display(label_df)

,Label,Record_Count,Percentage
1,BENIGN,2273097,80.300366
10,DoS Hulk,231073,8.162981
2,PortScan,158930,5.614427
0,DDoS,128027,4.522735
13,DoS GoldenEye,10293,0.363615
8,FTP-Patator,7938,0.280421
9,SSH-Patator,5897,0.208320
11,DoS slowloris,5796,0.204752
12,DoS Slowhttptest,5499,0.194260
3,Bot,1966,0.069452


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter

PROJECT_ROOT = Path("/home/darling/Projects/DeepShield")

DATA_DIR = (
    PROJECT_ROOT
    / "data/raw/CIC-IDS2017/MachineLearningCSV/MachineLearningCVE"
)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Dataset directory:", DATA_DIR)
print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file.name)

Dataset directory: /home/darling/Projects/DeepShield/data/raw/CIC-IDS2017/MachineLearningCSV/MachineLearningCVE
CSV files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [5]:
from collections import Counter

missing_counts = Counter()
total_rows = 0

print("Starting missing-value analysis...\n")

for file in csv_files:
    print(f"Scanning: {file.name}")

    file_missing = Counter()
    file_rows = 0

    for chunk in pd.read_csv(file, chunksize=50_000):
        chunk.columns = chunk.columns.str.strip()

        file_rows += len(chunk)
        file_missing.update(chunk.isna().sum().to_dict())

    total_rows += file_rows
    missing_counts.update(file_missing)

    total_missing = sum(file_missing.values())

    print(f"  Records scanned: {file_rows:,}")
    print(f"  Missing values: {total_missing:,}")

    columns_with_missing = {
        col: count
        for col, count in file_missing.items()
        if count > 0
    }

    if columns_with_missing:
        print("  Columns containing missing values:")
        for col, count in columns_with_missing.items():
            print(f"    {col}: {count:,}")
    else:
        print("  No missing values found.")

    print()

print("=" * 55)
print("OVERALL MISSING-VALUE SUMMARY")
print("=" * 55)

print(f"Total records scanned: {total_rows:,}")
print(f"Total missing values: {sum(missing_counts.values()):,}")

print("\nColumns with missing values:")

for col, count in missing_counts.most_common():
    if count > 0:
        print(f"{col}: {count:,}")

Starting missing-value analysis...

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Records scanned: 225,745
  Missing values: 4
  Columns containing missing values:
    Flow Bytes/s: 4

Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Records scanned: 286,467
  Missing values: 15
  Columns containing missing values:
    Flow Bytes/s: 15

Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Records scanned: 191,033
  Missing values: 28
  Columns containing missing values:
    Flow Bytes/s: 28

Scanning: Monday-WorkingHours.pcap_ISCX.csv
  Records scanned: 529,918
  Missing values: 64
  Columns containing missing values:
    Flow Bytes/s: 64

Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Records scanned: 288,602
  Missing values: 18
  Columns containing missing values:
    Flow Bytes/s: 18

Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Records scanned: 170,366
  Missing values: 20
  Columns containing missing v

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/home/darling/Projects/DeepShield")

DATA_DIR = (
    PROJECT_ROOT
    / "data/raw/CIC-IDS2017/MachineLearningCSV/MachineLearningCVE"
)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("CSV files found:", len(csv_files))

CSV files found: 8


In [4]:
import pandas as pd

missing_samples = []
missing_by_file = {}

columns_to_check = {
    "Flow Duration",
    "Total Length of Fwd Packets",
    "Total Length of Bwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Label"
}

for file in csv_files:
    count = 0

    for chunk in pd.read_csv(
        file,
        chunksize=50_000,
        usecols=lambda col: col.strip() in columns_to_check
    ):
        chunk.columns = chunk.columns.str.strip()

        missing_rows = chunk[chunk["Flow Bytes/s"].isna()]
        count += len(missing_rows)

        if len(missing_samples) < 10:
            remaining = 10 - len(missing_samples)
            missing_samples.extend(
                missing_rows.head(remaining).to_dict("records")
            )

    missing_by_file[file.name] = count

print("Missing values by file:")
for name, count in missing_by_file.items():
    if count:
        print(f"{name}: {count:,}")

print("\nFirst 10 affected records:")
display(pd.DataFrame(missing_samples))


Missing values by file:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 4
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 15
Friday-WorkingHours-Morning.pcap_ISCX.csv: 28
Monday-WorkingHours.pcap_ISCX.csv: 64
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 18
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 20
Tuesday-WorkingHours.pcap_ISCX.csv: 201
Wednesday-workingHours.pcap_ISCX.csv: 1,008

First 10 affected records:


,Flow Duration,Total Length of Fwd Packets,Total Length of Bwd Packets,Flow Bytes/s,Flow Packets/s,Label
0,0,0,0,NaN,inf,BENIGN
1,0,0,0,NaN,inf,BENIGN
2,0,0,0,NaN,inf,BENIGN
3,0,0,0,NaN,inf,BENIGN
4,0,0,0,NaN,inf,BENIGN
5,0,0,0,NaN,inf,BENIGN
6,0,0,0,NaN,inf,BENIGN
7,0,0,0,NaN,inf,BENIGN
8,0,0,0,NaN,inf,BENIGN
9,0,0,0,NaN,inf,BENIGN


In [6]:
from collections import Counter
import pandas as pd
import numpy as np

infinity_counts = Counter()
nan_counts = Counter()

zero_duration_rows = 0
total_rows = 0

for file in csv_files:
    print("Scanning:", file.name)

    for chunk in pd.read_csv(file, chunksize=50_000):
        chunk.columns = chunk.columns.str.strip()

        total_rows += len(chunk)

        numeric = chunk.select_dtypes(include="number")

        infinity_counts.update(
            np.isinf(numeric).sum().to_dict()
        )

        nan_counts.update(
            numeric.isna().sum().to_dict()
        )

        zero_duration_rows += chunk["Flow Duration"].eq(0).sum()

print("\n========== INFINITY SUMMARY ==========")

for col, count in infinity_counts.most_common():
    if count > 0:
        print(f"{col}: {count:,}")

print("\n========== ZERO DURATION SUMMARY ==========")
print(f"Zero-duration records: {zero_duration_rows:,}")

print("\n========== NUMERICAL NaN SUMMARY ==========")

for col, count in nan_counts.most_common():
    if count > 0:
        print(f"{col}: {count:,}")

print("\nTotal records scanned:", f"{total_rows:,}")

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv

========== INFINITY SUMMARY ==========
Flow Packets/s: 2,867
Flow Bytes/s: 1,509

========== ZERO DURATION SUMMARY ==========
Zero-duration records: 2,867

========== NUMERICAL NaN SUMMARY ==========
Flow Bytes/s: 1,358

Total records scanned: 2,830,743


In [7]:
from collections import Counter
import pandas as pd
import numpy as np

zero_duration_labels = Counter()

for file in csv_files:
    print("Scanning:", file.name)

    for chunk in pd.read_csv(
        file,
        chunksize=50_000,
        usecols=lambda col: col.strip() in {
            "Flow Duration",
            "Flow Bytes/s",
            "Flow Packets/s",
            "Label"
        }
    ):
        chunk.columns = chunk.columns.str.strip()

        mask = (
            (chunk["Flow Duration"] == 0)
            & (
                chunk["Flow Bytes/s"].isna()
                | np.isinf(chunk["Flow Bytes/s"])
            )
            & np.isinf(chunk["Flow Packets/s"])
        )

        labels = chunk.loc[mask, "Label"].astype("string").str.strip()

        zero_duration_labels.update(labels.value_counts().to_dict())

print("\n===== ZERO-DURATION RECORDS BY LABEL =====")

for label, count in zero_duration_labels.most_common():
    print(f"{label}: {count:,}")

print("\nTotal matching records:", sum(zero_duration_labels.values()))

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv

===== ZERO-DURATION RECORDS BY LABEL =====
BENIGN: 1,777
DoS Hulk: 949
PortScan: 126
Bot: 10
FTP-Patator: 3
DDoS: 2

Total matching records: 2867


In [8]:
import pandas as pd

duplicate_counts = {}
total_duplicate_rows = 0
total_rows_checked = 0

print("Starting duplicate analysis...\n")

for file in csv_files:
    print("Scanning:", file.name)

    seen_rows = set()
    file_duplicates = 0
    file_total = 0

    for chunk in pd.read_csv(file, chunksize=50_000):
        chunk.columns = chunk.columns.str.strip()

        hashes = pd.util.hash_pandas_object(
            chunk,
            index=False
        )

        for value in hashes:
            if value in seen_rows:
                file_duplicates += 1
            else:
                seen_rows.add(value)

        file_total += len(chunk)

    duplicate_counts[file.name] = file_duplicates
    total_duplicate_rows += file_duplicates
    total_rows_checked += file_total

    print(f"  Records: {file_total:,}")
    print(f"  Duplicate rows: {file_duplicates:,}\n")

print("=" * 50)
print("DUPLICATE SUMMARY")
print("=" * 50)
print(f"Total records checked: {total_rows_checked:,}")
print(f"Duplicate rows found: {total_duplicate_rows:,}")

Starting duplicate analysis...

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Records: 225,745
  Duplicate rows: 2,633

Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Records: 286,467
  Duplicate rows: 72,353

Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Records: 191,033
  Duplicate rows: 6,888

Scanning: Monday-WorkingHours.pcap_ISCX.csv
  Records: 529,918
  Duplicate rows: 26,935

Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Records: 288,602
  Duplicate rows: 35,630

Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Records: 170,366
  Duplicate rows: 6,066

Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
  Records: 445,909
  Duplicate rows: 24,065

Scanning: Wednesday-workingHours.pcap_ISCX.csv
  Records: 692,703
  Duplicate rows: 81,909

DUPLICATE SUMMARY
Total records checked: 2,830,743
Duplicate rows found: 256,479


In [10]:
duplicate_df = pd.DataFrame(
    duplicate_counts.items(),
    columns=["CSV File", "Duplicate Rows"]
)

duplicate_df["Duplicate Percentage"] = (
    duplicate_df["Duplicate Rows"]
    / duplicate_df["CSV File"].map(
        lambda name: next(
            file.stat().st_size for file in csv_files
            if file.name == name
        )
    )
)  # We'll replace this with the correct record-based calculation below.

In [11]:
duplicate_df = pd.DataFrame(
    duplicate_counts.items(),
    columns=["CSV File", "Duplicate Rows"]
)

display(duplicate_df)

print("Total duplicate rows:", f"{total_duplicate_rows:,}")
print(
    "Overall duplicate percentage:",
    f"{total_duplicate_rows / total_rows_checked * 100:.2f}%"
)

,CSV File,Duplicate Rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,2633
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,72353
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,6888
3,Monday-WorkingHours.pcap_ISCX.csv,26935
4,Thursday-WorkingHours-Afternoon-Infilteration....,35630
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,6066
6,Tuesday-WorkingHours.pcap_ISCX.csv,24065
7,Wednesday-workingHours.pcap_ISCX.csv,81909


Total duplicate rows: 256,479
Overall duplicate percentage: 9.06%


In [12]:
import pandas as pd

global_seen = set()

cross_dataset_duplicates = 0
total_scanned = 0

print("Checking duplicates across the entire dataset...\n")

for file in csv_files:
    print("Scanning:", file.name)

    file_duplicates = 0

    for chunk in pd.read_csv(file, chunksize=50_000):
        chunk.columns = chunk.columns.str.strip()

        hashes = pd.util.hash_pandas_object(
            chunk,
            index=False
        )

        for value in hashes:
            value = int(value)

            if value in global_seen:
                file_duplicates += 1
            else:
                global_seen.add(value)

        total_scanned += len(chunk)

    cross_dataset_duplicates += file_duplicates

    print(f"  Repeated rows: {file_duplicates:,}")

print("\n========== GLOBAL DUPLICATE SUMMARY ==========")

print(f"Total records scanned: {total_scanned:,}")
print(f"Repeated rows globally: {cross_dataset_duplicates:,}")

print(
    "Unique row hashes:",
    f"{len(global_seen):,}"
)

print(
    "Repeated percentage:",
    f"{cross_dataset_duplicates / total_scanned * 100:.2f}%"
)

Checking duplicates across the entire dataset...

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Repeated rows: 2,633
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Repeated rows: 73,878
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Repeated rows: 10,840
Scanning: Monday-WorkingHours.pcap_ISCX.csv
  Repeated rows: 27,131
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Repeated rows: 42,572
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Repeated rows: 13,677
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
  Repeated rows: 35,712
Scanning: Wednesday-workingHours.pcap_ISCX.csv
  Repeated rows: 101,938

========== GLOBAL DUPLICATE SUMMARY ==========
Total records scanned: 2,830,743
Repeated rows globally: 308,381
Unique row hashes: 2,522,362
Repeated percentage: 10.89%


In [13]:
from collections import Counter
import pandas as pd

feature_labels = {}
conflicting_hashes = set()

total_records = 0
conflicting_records = 0

print("Checking identical features with different labels...\n")

for file in csv_files:
    print("Scanning:", file.name)

    for chunk in pd.read_csv(file, chunksize=50_000):
        chunk.columns = chunk.columns.str.strip()

        labels = chunk["Label"].astype("string").str.strip()

        features = chunk.drop(columns=["Label"])

        hashes = pd.util.hash_pandas_object(
            features,
            index=False
        )

        for row_hash, label in zip(hashes, labels):
            row_hash = int(row_hash)
            label = str(label)

            total_records += 1

            previous_label = feature_labels.get(row_hash)

            if previous_label is None:
                feature_labels[row_hash] = label

            elif previous_label != label:
                conflicting_hashes.add(row_hash)
                conflicting_records += 1

print("\n========== LABEL CONFLICT SUMMARY ==========")

print("Total records checked:", f"{total_records:,}")

print(
    "Feature hashes with conflicting labels:",
    f"{len(conflicting_hashes):,}"
)

print(
    "Records with a label differing from the first observed label:",
    f"{conflicting_records:,}"
)

Checking identical features with different labels...

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv

========== LABEL CONFLICT SUMMARY ==========
Total records checked: 2,830,743
Feature hashes with conflicting labels: 698
Records with a label differing from the first observed label: 5,440


In [14]:
conflict_rows = []

print("Collecting records with potentially conflicting labels...\n")

for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=50_000):
        chunk.columns = chunk.columns.str.strip()

        hashes = pd.util.hash_pandas_object(
            chunk.drop(columns=["Label"]),
            index=False
        )

        mask = hashes.isin(conflicting_hashes)

        if mask.any():
            selected = chunk.loc[mask].copy()
            selected["Label"] = selected["Label"].astype("string").str.strip()
            conflict_rows.append(selected)

conflict_df = pd.concat(conflict_rows, ignore_index=True)

print("Potentially conflicting records collected:", len(conflict_df))

# Compare actual feature values, not just their hashes
feature_columns = [
    col for col in conflict_df.columns if col != "Label"
]

grouped = conflict_df.groupby(
    feature_columns,
    dropna=False,
    sort=False
)["Label"].agg(["nunique", "size"])

exact_conflicts = grouped[grouped["nunique"] > 1]

print("\n========== EXACT CONFLICT SUMMARY ==========")
print("Exact feature combinations with different labels:", len(exact_conflicts))
print("Records belonging to those exact conflict groups:", exact_conflicts["size"].sum())


Potentially conflicting records collected: 7020

========== EXACT CONFLICT SUMMARY ==========
Exact feature combinations with different labels: 698
Records belonging to those exact conflict groups: 7020
